# B2-024 — Practice p20

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** advanced · **Time budget:** 65 minutes  
**Concepts:** mixture-parameter-regression

## Task

Train an amortized regressor for mixture parameters, refine its predictions by least squares, and certify the pipeline against the fixed-initialization least-squares baseline; report the raw regressor honestly.

**Data.** `f, theta = capstone_data.train_rows("mixture")`: 800 functions sampled at the 32 points `capstone_data.MIXTURE_X`, float32 `(800, 32)`, and their canonical parameters `(800, 9)` in the layout `(w1, c1, s1, w2, c2, s2, w3, c3, s3)` sorted by center. Validation: `capstone_data.val_rows("mixture")` (200). The committed baseline fits each function by least squares with Adam (`lr = 0.02`, 500 steps) from a fixed initialization (centers 0.25/0.5/0.75, weights 1, median training width); it scores `0.0091` mean permutation-invariant test error at the frozen seed.

**Regressor.** `class Regressor(nn.Module)` with constructor `Regressor(train_values, hidden=128)` that registers buffers `mu`, `sd` (column mean and `torch.std` of `train_values`) and then creates `self.net = nn.Sequential(nn.Linear(32, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 9))`; `forward(values)` returns `self.net((values - self.mu) / self.sd)`. Construct with `torch.manual_seed(SEED)` then `model = Regressor(f)`; `optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`; exactly **1500** full-batch updates of `F.mse_loss(model(f), theta)` in train mode (Each update runs, in this order: `optimizer.zero_grad(set_to_none=True)`, forward, loss, `loss.backward()`, `optimizer.step()`; no scheduler, no gradient clipping, no accumulation, no early stopping.), recording the losses.
**Seam:** a forward pre-hook records `capstone_data.split_of("mixture", row)` for every row of the first training-mode call; it must be exactly `{"train"}`.

**Refinement.** Implement `mixture_function` and `canonicalize` (p11). Write `ls_refine(values, init_params, steps, lr=0.02)`: in float64, set `w`, `c` from the initial triples and `log_s = torch.log(widths.clamp_min(1e-3))`, all with `requires_grad=True`; `torch.optim.Adam([w, c, log_s], lr=lr)` (default betas and eps); each step minimizes `((mixture_function(MIXTURE_X, w, c, torch.exp(log_s)) - values) ** 2).mean(dim=1).sum()`; return `canonicalize` of the final `(w, c, exp(log_s))` triples as float32 `(N, 9)`.
**Pipeline:** `predict(values)` computes `canonicalize(model(values))` in eval mode under `torch.no_grad()`, then returns `ls_refine(values, that, steps=100)`.

**Validation report.** Report the mean permutation-invariant error (`capstone_data.perm_invariant_error_per_example`) on validation for (i) the raw regressor (canonicalized), (ii) the pipeline, and (iii) the baseline recipe (`capstone_data.mixture_ls_fit(values, capstone_data.mixture_baseline_init(200), steps=500)`). The raw regressor may lose to the per-function baseline; say so if it does, and explain why in at most three sentences.

**Evaluation protocol.** Fit only on rows from `capstone_data.train_rows("mixture")` (and any other rows this statement names), make every choice on `capstone_data.val_rows("mixture")`, and reach the locked test rows only through `capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="mixture")`, called exactly once, after all fitting and selection. Assert `capstone_data.test_call_count("mixture") == 0` immediately before that call and `== 1` after it.
Score the **pipeline** (`predict`).

**Certify.** The last training loss `<= 0.05 *` the first; the seam set is `{"train"}`; the pipeline's validation error is below the baseline recipe's validation error; the pipeline's test score is `< capstone_data.baseline_score("mixture").score`; the test-call protocol.
At the frozen seed the reference pipeline scores about `0.0046` against the baseline's `0.0091` (about half), and the raw regressor's validation error is about `0.024`, worse than the baseline's `0.0087`; the 100 refinement steps were checked to clear the baseline with this slack, so no change to the plan's step count was needed.

**Required answer-check assertions.** Every certificate item, `len(losses) == 1500`, and the pipeline's validation output shape `(200, 9)` with every width positive.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

TRAIN_STEPS = 1500
REFINE_STEPS = 100
REFINE_LR = 0.02

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

On a Colab L4, scale the same regressor and refinement code with a replacement config: simulate many more training functions (`capstone_data.simulate("mixture", 500000, seed=4)`), widen the network, train with mini-batches, and keep the 100-step refinement (in float32 on the GPU, which is fast enough there).

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "simulated_functions": 500000, "hidden": 512,
             "steps": 30000, "batch_size": 4096, "lr": 1e-3, "refine_steps": 100, "refine_dtype": torch.float32}
```

**What to observe.** Whether the raw regressor overtakes the per-function baseline once it has hundreds of thousands of training functions; how much refinement still adds; and the effect of `bfloat16` autocast on the regressor's parameter precision (refinement should stay in `float32`).

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.